# 🤖 Train Your Custom Mindneox AI Model

This notebook helps you fine-tune a language model using your collected conversation data.

**Requirements:**
- Google Colab with GPU (free)
- Your `training_data.jsonl` file
- ~1000+ conversation pairs for best results

**What you'll get:**
- Custom fine-tuned model trained on your data
- Model uploaded to Hugging Face
- Ready to deploy in your chatbot

## Step 1: Setup Environment

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_hub

# Optional: Install Unsloth for 2x faster training (recommended)
!pip install -q unsloth

## Step 2: Upload Your Training Data

Upload your `training_data.jsonl` file using the file upload button on the left sidebar.

In [ ]:
from google.colab import files
import json

# Upload training data
print("📤 Upload your training_data.jsonl file:")
uploaded = files.upload()

# Verify data
with open('training_data.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]
    print(f"\n✅ Loaded {len(data)} training examples")
    print(f"\nExample:")
    print(json.dumps(data[0], indent=2))

## Step 3: Prepare Dataset

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset('json', data_files='training_data.jsonl', split='train')

# Split into train/validation (90/10)
dataset = dataset.train_test_split(test_size=0.1)

print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['test'])}")

# Format for training
def format_prompt(example):
    return {
        "text": f"<|system|>\nYou are Mindneox AI, a helpful assistant.</s>\n<|user|>\n{example['prompt']}</s>\n<|assistant|>\n{example['completion']}</s>"
    }

dataset = dataset.map(format_prompt)
print("\n✅ Dataset prepared!")

## Step 4: Load Base Model

Choose your base model:
- **TinyLlama-1.1B** - Fast, lightweight (recommended for free Colab)
- **Phi-2** - Better quality, slower
- **Mistral-7B** - Best quality, requires Pro Colab

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

# Choose your base model
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Change if needed

print(f"📥 Loading {MODEL_NAME}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with 4-bit quantization (saves memory)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

print("✅ Model loaded!")

## Step 5: Configure LoRA (Efficient Fine-tuning)

In [ ]:
# LoRA configuration - trains only small adapters (saves memory & time)
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total params: {total_params:,}")

## Step 6: Train the Model

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./mindneox-custom",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    save_steps=100,
    logging_steps=10,
    save_total_limit=2,
    warmup_steps=50,
    evaluation_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Starting training...")
print("This will take 15-30 minutes depending on dataset size.")
print("\n" + "="*50)

# Train!
trainer.train()

print("\n" + "="*50)
print("✅ Training complete!")

## Step 7: Test Your Model

In [ ]:
# Test the fine-tuned model
def generate_response(prompt):
    formatted_prompt = f"<|system|>\nYou are Mindneox AI, a helpful assistant.</s>\n<|user|>\n{prompt}</s>\n<|assistant|>\n"
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.95,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response
    response = response.split("<|assistant|>")[-1].strip()
    return response

# Test with sample prompts
test_prompts = [
    "Hello, how are you?",
    "What is artificial intelligence?",
    "Can you help me with coding?"
]

print("🧪 Testing your custom model:\n")
for prompt in test_prompts:
    print(f"User: {prompt}")
    response = generate_response(prompt)
    print(f"AI: {response}")
    print("-" * 50)

## Step 8: Save and Upload to Hugging Face

In [ ]:
from huggingface_hub import login, HfApi

# Login to Hugging Face
print("🔐 Login to Hugging Face:")
login()

# Save model locally
model.save_pretrained("./mindneox-custom-final")
tokenizer.save_pretrained("./mindneox-custom-final")

print("\n💾 Model saved locally!")

# Upload to Hugging Face
model_name = input("Enter model name (e.g., your-username/mindneox-custom): ")

print(f"\n📤 Uploading to {model_name}...")
model.push_to_hub(model_name)
tokenizer.push_to_hub(model_name)

print("\n✅ Model uploaded to Hugging Face!")
print(f"🌐 View at: https://huggingface.co/{model_name}")
print("\n🚀 You can now use this model in your chatbot!")

## Step 9: Download Model (Optional)

Download the model to use locally or convert to GGUF format.

In [ ]:
# Download model files
from google.colab import files
import shutil

# Create zip file
shutil.make_archive('mindneox-custom-model', 'zip', './mindneox-custom-final')

print("📦 Downloading model...")
files.download('mindneox-custom-model.zip')

print("✅ Download complete!")
print("\nTo convert to GGUF format (for llama.cpp):")
print("1. Extract the zip file")
print("2. Use llama.cpp convert script")
print("3. Quantize with llama.cpp")

## 🎉 Congratulations!

You've successfully trained your custom Mindneox AI model!

### Next Steps:

1. **Deploy your model:**
   - Update your backend to use the new model
   - Set `HF_INFERENCE_MODEL` to your model name

2. **Collect more data:**
   - Continue collecting conversations
   - Retrain monthly for improvements

3. **Optimize:**
   - Convert to GGUF for faster inference
   - Quantize to reduce size

### Resources:
- [Hugging Face Docs](https://huggingface.co/docs)
- [PEFT Documentation](https://huggingface.co/docs/peft)
- [TRL Documentation](https://huggingface.co/docs/trl)

Happy training! 🚀